# 📄 Document Intelligence Demo: PDF → Structured Product Catalog

A self-contained notebook that walks through the complete **Intelligent Document Processing** pipeline on Databricks — from raw PDFs to a structured, queryable product catalog with quality evaluation.

**What this notebook does, end to end:**
1. Creates a Unity Catalog catalog, schema, and volume
2. Parses product manual PDFs using `ai_parse_document`
3. Extracts 14 structured fields using `ai_extract` (v2)
4. Flattens the results into a typed Delta table
5. Evaluates extraction quality using MLflow 3 GenAI evaluation

**Use case:** Power tool manuals (Bosch, Makita, DeWalt, Milwaukee) → structured product catalog ready for SQL queries and natural language exploration.

> 📖 Read the companion blog post: [Intelligent Document Processing for Data Extraction](https://community.databricks.com/t5/technical-blog/intelligent-document-processing-for-data-extraction-transforming/ba-p/153847)

## 🔧 Prerequisites

- Databricks workspace with **Unity Catalog** enabled
- Serverless compute or a cluster running **DBR 14.3 ML** or later
- `CREATE CATALOG` privilege (or an existing catalog you can write to)
- PDF files to process — sample power tool manuals are in `productmanuals/` in this repository

## ⚙️ Configuration

Edit the cell below **before running** any other cells.

In [ ]:
# ── Edit these values for your workspace ──────────────────────────────────────
CATALOG      = 'my_catalog'           # Unity Catalog catalog name
SCHEMA       = 'my_schema'            # Schema (database) name
VOLUME       = 'product_manuals'      # Volume name (created if it does not exist)
TABLE_PREFIX = 'demo'                 # Prefix applied to all Delta tables created here
EXPERIMENT   = '/Shared/product_manuals_extraction_eval'  # MLflow experiment path

# ── Derived paths — no need to edit ───────────────────────────────────────────
volume_path = f'/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}'
pdfs_path   = f'{volume_path}/productmanuals'

print(f'catalog      : {CATALOG}')
print(f'schema       : {SCHEMA}')
print(f'volume path  : {volume_path}')
print(f'pdfs path    : {pdfs_path}')
print(f'table prefix : {TABLE_PREFIX}')
print(f'experiment   : {EXPERIMENT}')

## 🏗️ Step 0: Create Catalog, Schema & Volume

Creates (or verifies) the Unity Catalog objects needed by the pipeline.

In [ ]:
# Create catalog if it does not exist
if not spark.sql(f"SHOW CATALOGS LIKE '{CATALOG}'").count():
    spark.sql(f'CREATE CATALOG `{CATALOG}`')
    print(f'Created catalog: {CATALOG}')
else:
    print(f'Catalog already exists: {CATALOG}')

spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`')
print(f'Schema ready: {CATALOG}.{SCHEMA}')

spark.sql(f'CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME}`')
print(f'Volume ready: {volume_path}')

## 📁 Step 0b: Upload Sample PDF Files

The parsing step reads PDFs from the `productmanuals/` subfolder of your volume.
The cell below checks whether files are present.

**If the directory is empty,** upload PDF files to `pdfs_path` (printed in the Configuration cell).
Sample power tool manuals live in `productmanuals/` in this repository — use the copy snippet in the next cell.

In [ ]:
# Check that PDF files are present in the volume
try:
    files     = dbutils.fs.ls(pdfs_path)
    pdf_files = [f for f in files if f.name.lower().endswith('.pdf')]
    if pdf_files:
        print(f'Found {len(pdf_files)} PDF file(s) in {pdfs_path}:')
        for f in pdf_files:
            print(f'  {f.name}  ({f.size:,} bytes)')
    else:
        print(f'WARNING: No PDF files found in {pdfs_path}')
        print(f'Please upload .pdf files to: {pdfs_path}')
except Exception as e:
    if 'FileNotFoundException' in str(e):
        dbutils.fs.mkdirs(pdfs_path)
        print(f'Created directory: {pdfs_path}')
        print(f'Please upload .pdf files to: {pdfs_path}')
    else:
        raise e

## 📄 Step 1: Parse PDFs with `ai_parse_document`

[`ai_parse_document`](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_parse_document)
reads each PDF as binary data and returns a structured JSON object containing:

- Extracted **text** (with layout context)
- **Tables** in structured form
- **Figures** with AI-generated descriptions

The result is stored as a Delta table. A flat `document` string is also extracted
for use in the evaluation step.

> ⏱️ Expect **1–3 minutes per PDF** depending on document size.

In [ ]:
spark.sql(f'''
    CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.`{TABLE_PREFIX}_productmanuals_parsed`
    COMMENT 'Parsed product manual PDFs with layout-aware text, tables, and figure descriptions'
    AS
    WITH raw AS (
        SELECT
            _metadata.file_path  AS path,
            _metadata.file_name  AS file_name,
            _metadata.file_size  AS file_size,
            ai_parse_document(
                content,
                map('version', '2.0', 'descriptionElementTypes', '*')
            ) AS parsed
        FROM READ_FILES(
            '{pdfs_path}',
            format => 'binaryFile',
            fileNamePattern => '*.pdf'
        )
    )
    SELECT *, parsed:document::string AS document FROM raw
''')
print(f'Created: {CATALOG}.{SCHEMA}.{TABLE_PREFIX}_productmanuals_parsed')

In [ ]:
display(
    spark.table(f'{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_productmanuals_parsed')
    .select('file_name', 'file_size', 'document')
)

## 🔍 Step 2: Extract Structured Fields with `ai_extract`

[`ai_extract`](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_extract) (v2)
takes the parsed document and a declarative JSON **schema** to pull out specific fields.
Field descriptions are extraction hints that handle:

- **Vendor terminology differences** (e.g. "max torque", "tightening torque" → `max_torque_nm`)
- **Unit conversions** (lbs → kg)
- **Multi-language documents** (restrict extraction to English sections)

> ⏱️ Expect **1–3 minutes per PDF**.

In [ ]:
import json

# ── Extraction schema ──────────────────────────────────────────────────────────
# Each field's description acts as a prompt hint for the LLM extractor.
schema_dict = {
    'manufacturer': {
        'type': 'string',
        'description': 'Brand or manufacturer name, e.g. Bosch, Makita, BLACK+DECKER',
    },
    'model_number': {
        'type': 'string',
        'description': 'Product model number or identifier, e.g. GSR 18V-65, BCD382, DF033D',
    },
    'product_name': {
        'type': 'string',
        'description': 'Full product name or description, e.g. Cordless Drill/Driver, 20V MAX Cordless Drill',
    },
    'product_type': {
        'type': 'string',
        'description': 'Type of tool: drill, drill/driver, hammer drill, impact driver, etc.',
    },
    'rated_voltage_v': {
        'type': 'number',
        'description': 'Rated or nominal voltage in volts',
    },
    'max_torque_nm': {
        'type': 'number',
        'description': (
            'Maximum torque in Newton-meters (Nm). Use the hard screwdriving value if both hard and soft '
            'are given. May appear as max torque, tightening torque, or fastening torque.'
        ),
    },
    'no_load_speed_low_rpm': {
        'type': 'number',
        'description': (
            'No-load speed for gear 1 or low speed setting in RPM. Use the max value if a range is given. '
            'May appear as speed setting 1 or low gear.'
        ),
    },
    'no_load_speed_high_rpm': {
        'type': 'number',
        'description': (
            'No-load speed for gear 2 or high speed setting in RPM. Use the max value if a range is given. '
            'May appear as speed setting 2 or high gear.'
        ),
    },
    'chuck_capacity_mm': {
        'type': 'string',
        'description': (
            'Chuck capacity range in mm, e.g. 1.5-13 or 10. '
            'Look for chuck size, collet capacity, or clamping range.'
        ),
    },
    'max_drilling_diameter_wood_mm': {
        'type': 'number',
        'description': (
            'Maximum drilling diameter in wood in mm. '
            'May appear as drilling capacity in wood or max bore diameter wood.'
        ),
    },
    'max_drilling_diameter_steel_mm': {
        'type': 'number',
        'description': (
            'Maximum drilling diameter in steel in mm. '
            'May appear as drilling capacity in steel or max bore diameter steel.'
        ),
    },
    'weight_kg': {
        'type': 'number',
        'description': (
            'Weight of the tool in kg without battery. '
            'If given in lbs, convert to kg (1 lb = 0.4536 kg). '
            'Use minimum value if a range is given.'
        ),
    },
    'compatible_batteries': {
        'type': 'string',
        'description': (
            'List of compatible battery packs or battery model numbers, comma-separated. '
            'Look in accessories, battery, or recommended sections.'
        ),
    },
    'compatible_chargers': {
        'type': 'string',
        'description': (
            'List of compatible chargers or charger model numbers, comma-separated. '
            'Look in accessories, charger, or recommended sections.'
        ),
    },
}

instructions = (
    'Extract product specifications from this power tool manual. '
    'Focus only on the English language sections. '
    'All extracted values (including product_name) must be in English. '
    'Look for technical data in specification tables, feature lists, and product descriptions '
    'throughout the entire document. If a specification is mentioned anywhere in the document '
    '(not just in tables), extract it. If multiple models are listed, extract the primary model.'
)

schema_json = json.dumps(schema_dict)
print(f'Schema ready: {len(schema_dict)} fields defined.')

In [ ]:
spark.sql(f'''
    CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.`{TABLE_PREFIX}_productmanuals_extract`
    COMMENT 'AI-extracted product specifications via ai_extract v2'
    AS
    SELECT
        path,
        file_name,
        file_size,
        ai_extract(
            parsed,
            '{schema_json}',
            map('version', '2.0', 'instructions', '{instructions}')
        ) AS ai_result
    FROM `{CATALOG}`.`{SCHEMA}`.`{TABLE_PREFIX}_productmanuals_parsed`
''')
print(f'Created: {CATALOG}.{SCHEMA}.{TABLE_PREFIX}_productmanuals_extract')

In [ ]:
display(
    spark.table(f'{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_productmanuals_extract')
    .select('file_name', 'ai_result')
)

## 🧹 Step 3: Flatten & Type-Cast into a Clean Table

The `ai_result` column is a nested JSON struct. This step flattens it into strongly typed
columns with descriptive comments.

The column comments matter — downstream tools like **Genie Space** use them to generate
more accurate SQL queries from natural language.

In [ ]:
import pyspark.sql.functions as F

df_extract = spark.table(f'{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_productmanuals_extract')

df_processed = df_extract.select(
    F.col('file_name'),
    F.expr('ai_result:response.manufacturer::STRING').alias('manufacturer'),
    F.expr('ai_result:response.model_number::STRING').alias('model_number'),
    F.expr('ai_result:response.product_name::STRING').alias('product_name'),
    F.expr('ai_result:response.product_type::STRING').alias('product_type'),
    F.expr('ai_result:response.rated_voltage_v::DOUBLE').alias('rated_voltage_v'),
    F.expr('ai_result:response.max_torque_nm::DOUBLE').alias('max_torque_nm'),
    F.expr('ai_result:response.no_load_speed_low_rpm::INT').alias('no_load_speed_low_rpm'),
    F.expr('ai_result:response.no_load_speed_high_rpm::INT').alias('no_load_speed_high_rpm'),
    F.expr('ai_result:response.chuck_capacity_mm::STRING').alias('chuck_capacity_mm'),
    F.expr('ai_result:response.max_drilling_diameter_wood_mm::DOUBLE').alias('max_drilling_diameter_wood_mm'),
    F.expr('ai_result:response.max_drilling_diameter_steel_mm::DOUBLE').alias('max_drilling_diameter_steel_mm'),
    F.expr('ai_result:response.weight_kg::DOUBLE').alias('weight_kg'),
    F.expr('ai_result:response.compatible_batteries::STRING').alias('compatible_batteries'),
    F.expr('ai_result:response.compatible_chargers::STRING').alias('compatible_chargers'),
)

df_processed.write.mode('overwrite').saveAsTable(
    f'{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_productmanuals_processed'
)
print(f'Created: {CATALOG}.{SCHEMA}.{TABLE_PREFIX}_productmanuals_processed')

In [ ]:
# Add descriptive column comments — used by Genie Space for natural language SQL generation
column_comments = {
    'file_name':                         'Original PDF file name.',
    'manufacturer':                      'Brand or manufacturer name.',
    'model_number':                      'Product model number or identifier.',
    'product_name':                      'Full product name or description.',
    'product_type':                      'Type of tool (drill, drill/driver, hammer drill, etc.).',
    'rated_voltage_v':                   'Rated voltage in volts.',
    'max_torque_nm':                     'Maximum torque in Newton-meters.',
    'no_load_speed_low_rpm':             'No-load speed for low gear in RPM.',
    'no_load_speed_high_rpm':            'No-load speed for high gear in RPM.',
    'chuck_capacity_mm':                 'Chuck capacity range in mm.',
    'max_drilling_diameter_wood_mm':     'Maximum drilling diameter in wood in mm.',
    'max_drilling_diameter_steel_mm':    'Maximum drilling diameter in steel in mm.',
    'weight_kg':                         'Weight of the tool in kg (without battery).',
    'compatible_batteries':              'Compatible battery packs.',
    'compatible_chargers':               'Compatible chargers.',
}
for col, comment in column_comments.items():
    spark.sql(
        f"ALTER TABLE `{CATALOG}`.`{SCHEMA}`.`{TABLE_PREFIX}_productmanuals_processed` "
        f"ALTER COLUMN {col} COMMENT '{comment}'"
    )
print('Column comments applied.')

In [ ]:
display(spark.table(f'{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_productmanuals_processed'))

## 📊 Step 4: Evaluate Extraction Quality with MLflow

Without ground-truth labels, we measure quality using **MLflow 3 GenAI evaluation** —
a mix of fast code scorers and LLM-as-judge scorers:

| Scorer | Type | What it measures |
|--------|------|-----------------|
| `completeness` | Code | Fraction of the 14 fields that are non-null |
| `format_validator` | Code | Numeric fields within physically plausible ranges |
| `english_compliance` | LLM judge | All extracted values are in English |
| `extraction_quality` | LLM judge | Values look like real power-tool specifications |

Results are logged to an MLflow experiment — refine the schema, re-run, and compare
metrics across runs to track quality improvements.

In [ ]:
import mlflow
import mlflow.genai
from mlflow.genai.scorers import Guidelines, scorer
from mlflow.entities import Feedback

mlflow.set_tracking_uri('databricks')
mlflow.set_experiment(EXPERIMENT)
print(f'MLflow experiment: {EXPERIMENT}')

In [ ]:
# ── Load processed and parsed tables ──────────────────────────────────────────
df_processed = spark.table(f'{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_productmanuals_processed')
df_parsed    = spark.table(f'{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_productmanuals_parsed')

df_eval = df_processed.join(
    df_parsed.select('file_name', 'document'),
    on='file_name',
    how='inner',
)
display(df_eval.limit(5))

In [ ]:
# ── Build evaluation dataset ───────────────────────────────────────────────────
EXTRACTION_FIELDS = [
    'manufacturer', 'model_number', 'product_name', 'product_type',
    'rated_voltage_v', 'max_torque_nm', 'no_load_speed_low_rpm',
    'no_load_speed_high_rpm', 'chuck_capacity_mm',
    'max_drilling_diameter_wood_mm', 'max_drilling_diameter_steel_mm',
    'weight_kg', 'compatible_batteries', 'compatible_chargers',
]

rows = df_eval.collect()

eval_data = []
for row in rows:
    outputs = {field: row[field] for field in EXTRACTION_FIELDS}
    outputs_str = '\n'.join(f'  {k}: {v}' for k, v in outputs.items())
    eval_data.append({
        'inputs': {
            'file_name':   row['file_name'],
            'source_text': (row['document'] or '')[:5000],
        },
        'outputs': {
            'response':         outputs_str,
            'extracted_fields': outputs,
        },
    })

print(f'Evaluation dataset: {len(eval_data)} record(s)')
for r in eval_data:
    print(f"  - {r['inputs']['file_name']}")

In [ ]:
# ── Code-based scorers ─────────────────────────────────────────────────────────

@scorer(aggregations=['mean', 'min', 'max'])
def completeness(outputs):
    # Fraction of the 14 extraction fields that are non-null
    fields = outputs.get('extracted_fields', {})
    total  = len(EXTRACTION_FIELDS)
    filled = sum(1 for f in EXTRACTION_FIELDS if fields.get(f) is not None)
    ratio  = filled / total if total > 0 else 0.0
    return Feedback(
        value=round(ratio, 2),
        rationale=f'{filled}/{total} fields extracted ({ratio * 100:.0f}%)',
    )


@scorer
def format_validator(outputs):
    # Numeric fields within physically plausible ranges for a cordless power tool
    import re
    fields  = outputs.get('extracted_fields', {})
    checks  = {
        'rated_voltage_v':               (1,   60,    'Voltage should be 1-60 V'),
        'max_torque_nm':                 (1,   300,   'Torque should be 1-300 Nm'),
        'weight_kg':                     (0.3, 15,    'Weight should be 0.3-15 kg'),
        'no_load_speed_low_rpm':         (50,  5000,  'Low speed should be 50-5000 RPM'),
        'no_load_speed_high_rpm':        (100, 10000, 'High speed should be 100-10000 RPM'),
        'max_drilling_diameter_wood_mm': (5,   100,   'Wood drilling should be 5-100 mm'),
        'max_drilling_diameter_steel_mm':(3,   50,    'Steel drilling should be 3-50 mm'),
    }
    issues      = []
    valid_count = 0
    for field, (lo, hi, desc) in checks.items():
        val = fields.get(field)
        if val is None:
            continue
        try:
            num = float(val)
            if lo <= num <= hi:
                valid_count += 1
            else:
                issues.append(f'{field}={num} out of range ({desc})')
        except (ValueError, TypeError):
            issues.append(f'{field}={val} is not numeric')
    chuck = fields.get('chuck_capacity_mm')
    if chuck is not None:
        if re.match(r'^\d+(\.\d+)?(-\d+(\.\d+)?)?(\s*mm)?$', str(chuck).strip()):
            valid_count += 1
        else:
            issues.append(f"chuck_capacity_mm='{chuck}' unexpected format")
    if issues:
        return Feedback(name='format_issues', value=False,
                        rationale=f"Issues: {'; '.join(issues)}")
    return Feedback(name='format_issues', value=True,
                    rationale=f'All {valid_count} present numeric fields within expected ranges')

In [ ]:
# ── LLM judge scorers ──────────────────────────────────────────────────────────

english_check = Guidelines(
    name='english_compliance',
    guidelines=[
        'All extracted values in the response must be in English.',
        'No German, French, Japanese, or other non-English language terms should appear in any field value.',
        'Product names, descriptions, and all text fields must use English words only.',
        'Brand names (e.g. Bosch, Makita) and model numbers (e.g. GSR 18V-65) are acceptable as-is.',
    ],
)

extraction_quality = Guidelines(
    name='extraction_quality',
    guidelines=[
        'The manufacturer must be a real, well-known power tool brand name.',
        'The model_number must look like a real product identifier (alphanumeric pattern, not a sentence).',
        'The product_type must be a standard tool category: drill, drill/driver, hammer drill, or impact driver.',
        'Numeric specifications like voltage, torque, and speed must be plausible for a cordless power tool.',
        'The response should not contain hallucinated or fabricated values that contradict power tool specifications.',
    ],
)

print('Scorers ready: completeness, format_validator, english_compliance, extraction_quality')

In [ ]:
# ── Run evaluation ─────────────────────────────────────────────────────────────
results = mlflow.genai.evaluate(
    data=eval_data,
    scorers=[completeness, format_validator, english_check, extraction_quality],
)

print(f'Run ID: {results.run_id}')
print('\nAggregate metrics:')
for metric, value in sorted(results.metrics.items()):
    print(f'  {metric}: {value}')
print(f'\nOpen the MLflow experiment for per-document scores and rationales:')
print(f'  {EXPERIMENT}')

## ✅ Next Steps

You've completed the end-to-end demo. Here's what to explore next:

| Next Step | Description |
|-----------|-------------|
| **Iterate on the schema** | Refine field descriptions and re-run Steps 2–4 to compare metrics across MLflow runs |
| **Add more documents** | Upload additional PDFs to the volume — `ai_parse_document` and `ai_extract` scale to thousands |
| **Deploy the full pipeline** | See `databricks_etl/` for the production Lakeflow Spark Declarative Pipeline with incremental processing |
| **Add a Genie Space** | Query the processed table with natural language — column comments drive accurate SQL generation |
| **Add a Knowledge Assistant** | Index the raw PDFs for open-ended, citation-grounded Q&A |
| **Deploy the Databricks App** | See `databricks_app/` for the FastAPI + React app that ties uploads, queries, and the supervisor agent together |

> 📖 Full deployment guide: [README](../README.md)